# Phase 4.3: Ranking Models

Turn the Random Forest and Two-Tower into rankers: train on visited (1) vs unvisited (0) cities,
score every candidate city for each test user, and compare with the baselines using Recall@10, NDCG@10 and MRR@10.

In [1]:
import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
from config import *
from src.data.cache import DataCache
from src.evaluation.harness import RecommendationHarness
from src.features.ranking_features import (
    USER_FEATURES, CITY_FEATURES, PAIR_FEATURES, WEATHER_FEATURES,
    city_features, user_features, trip_months, city_month_weather, build_pair_features,
)
from src.features.weather_features import WeatherFeatureBuilder
from src.models.baselines import PopularityBaseline
from src.models.rankers import RandomForestRanker, TwoTowerRanker

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
reviews = cache.load_dataframe('reviews_cross_city')

In [2]:
# Same test users as Phase 1; validation users are carved out of train for tuning
harness = RecommendationHarness(
    reviews,
    businesses,
    test_size=PHASE1_TEST_SIZE,
    val_size=PHASE1_VAL_SIZE,
    random_state=RANDOM_STATE,
    n_negatives=PHASE1_NEGATIVES_PER_POSITIVE
)
train_reviews, test_reviews = harness.create_train_test_split()
all_cities = sorted(businesses['city'].unique())
train_labels = harness.get_training_labels(all_cities)
val_pairs = harness.get_validation_pairs()
test_pairs = harness.get_test_pairs()

print(f"Users: train {len(harness.train_users):,} · val {len(harness.val_users):,} · test {len(harness.test_users):,}")
print(f"Training rows: {len(train_labels):,}")

Users: train 6,886 · val 984 · test 1,968
Training rows: 34,430


In [3]:
# Features: user history, city stats, home-to-city distance, trip-month weather
history = pd.concat([train_reviews, harness.val_reviews, test_reviews])
cities = city_features(businesses)
users = user_features(history, businesses, cities)
months = trip_months(reviews, businesses, harness.targets)
weather = city_month_weather(cities, WeatherFeatureBuilder(cache_dir=WEATHER_CACHE))
train_df = build_pair_features(train_labels, users, cities, months, weather)

print(f"Weather: {len(weather):,} city-months")
train_df[USER_FEATURES + CITY_FEATURES + PAIR_FEATURES + WEATHER_FEATURES].describe().T[['mean', 'min', 'max']].round(2)

Weather: 3,372 city-months


,mean,min,max
user_avg_stars,3.81,1.00,5.00
user_std_stars,1.10,0.00,2.83
user_log_reviews,2.67,0.69,7.50
user_num_cities,4.58,1.00,90.00
home_lat,35.84,27.71,53.63
home_lon,-88.58,-119.86,-74.71
city_log_businesses,5.57,3.93,9.59
city_avg_stars,3.54,2.88,4.16
city_log_reviews,9.06,6.75,13.75
lat,36.93,27.71,53.63


In [4]:
def scorer(model):
    """Turn a fitted model into a score function over (user_id, city) candidate rows."""
    return lambda candidates: model.score(build_pair_features(candidates, users, cities, months, weather))

def evaluate(score_fn, pairs=test_pairs):
    return harness.evaluate(pairs, all_cities, score_fn, k=PHASE1_EVAL_K)

popularity = PopularityBaseline(businesses, history)
popularity_scores = {city: popularity.score_city(city) for city in all_cities}
rng = np.random.default_rng(RANDOM_STATE)

results = {
    'Random': evaluate(lambda c: rng.random(len(c))),
    'Popularity': evaluate(lambda c: c['city'].map(popularity_scores).to_numpy()),
}
pd.DataFrame(results).T.round(3)

,recall@10,ndcg@10,mrr@10,num_queries
Random,0.038,0.018,0.013,1968.0
Popularity,0.358,0.180,0.126,1968.0


In [5]:
# Random Forest: pick tree depth on validation users, then train with and without weather
base = USER_FEATURES + CITY_FEATURES + PAIR_FEATURES
with_weather = base + WEATHER_FEATURES

val_recall = {}
for depth in [8, 15]:
    rf = RandomForestRanker(base, RF_N_ESTIMATORS, depth, RF_RANDOM_STATE).fit(train_df)
    val_recall[depth] = evaluate(scorer(rf), val_pairs)[f'recall@{PHASE1_EVAL_K}']
best_depth = max(val_recall, key=val_recall.get)
print(f"Validation Recall@10 by depth: {val_recall} -> depth {best_depth}")

rf_base = RandomForestRanker(base, RF_N_ESTIMATORS, best_depth, RF_RANDOM_STATE).fit(train_df)
rf_weather = RandomForestRanker(with_weather, RF_N_ESTIMATORS, best_depth, RF_RANDOM_STATE).fit(train_df)
results['Random Forest'] = evaluate(scorer(rf_base))
results['Random Forest + weather'] = evaluate(scorer(rf_weather))

pd.Series(rf_weather.feature_importance()).sort_values(ascending=False).round(3)

Validation Recall@10 by depth: {8: np.float64(0.6941056910569106), 15: np.float64(0.6910569105691057)} -> depth 8


distance_km            0.389
city_log_reviews       0.178
city_log_businesses    0.151
temp_diff              0.093
home_lat               0.033
home_lon               0.032
lon                    0.029
lat                    0.026
city_avg_stars         0.022
user_log_reviews       0.014
user_num_cities        0.009
user_std_stars         0.007
city_temp              0.007
city_precip            0.006
user_avg_stars         0.004
dtype: float64

In [6]:
# Two-Tower: user tower and city tower, best epoch picked on validation Recall@10
val_fn = lambda model: evaluate(scorer(model), val_pairs)[f'recall@{PHASE1_EVAL_K}']

tt_base = TwoTowerRanker(
    USER_FEATURES, CITY_FEATURES, TOWER_HIDDEN_DIM, TOWER_EMBEDDING_DIM, TOWER_LEARNING_RATE, epochs=TOWER_EPOCHS
).fit(train_df, val_fn)
tt_weather = TwoTowerRanker(
    USER_FEATURES, CITY_FEATURES + ['city_temp', 'city_precip'], TOWER_HIDDEN_DIM, TOWER_EMBEDDING_DIM, TOWER_LEARNING_RATE, epochs=TOWER_EPOCHS
).fit(train_df, val_fn)
results['Two-Tower'] = evaluate(scorer(tt_base))
results['Two-Tower + weather'] = evaluate(scorer(tt_weather))

print('Validation Recall@10 per epoch:', [round(v, 3) for v in tt_base.history])

Validation Recall@10 per epoch: [np.float64(0.548), np.float64(0.559), np.float64(0.584), np.float64(0.599), np.float64(0.593), np.float64(0.614), np.float64(0.613), np.float64(0.62), np.float64(0.611), np.float64(0.61), np.float64(0.627), np.float64(0.622), np.float64(0.629), np.float64(0.623), np.float64(0.633), np.float64(0.623), np.float64(0.628), np.float64(0.621), np.float64(0.629), np.float64(0.629)]


In [7]:
table = pd.DataFrame(results).T.drop(columns='num_queries').round(3)
cache.save_metadata({'test_users': len(test_pairs), 'results': table.to_dict(orient='index')}, 'ranking_results')
print(f"Test users: {len(test_pairs):,}")
table

Test users: 1,968


,recall@10,ndcg@10,mrr@10
Random,0.038,0.018,0.013
Popularity,0.358,0.180,0.126
Random Forest,0.723,0.541,0.484
Random Forest + weather,0.720,0.539,0.483
Two-Tower,0.634,0.432,0.370
Two-Tower + weather,0.633,0.435,0.373
